# Análisis estratégico y modelado predictivo: Wingspan

**Proyecto final — Diplomado en Transformación Digital Empresarial**  
**Autora:** Marcela Cadena  
**Fecha:** Junio 2026

---

## Pregunta de investigación

> ¿Qué atributos de las cartas de Wingspan (hábitat, costo de comida, tipo de nido, puntos base) permiten predecir el tipo de motor de juego que una carta potencia?

**Motivación estratégica:** En Wingspan, construir el motor correcto desde las primeras cartas es la clave del éxito. Este análisis busca identificar, usando Machine Learning, cuáles características estructurales de una carta señalan qué tipo de motor activará durante la partida — alimento, huevos o robo de cartas — permitiendo tomar mejores decisiones en el *draft* inicial.

---

**Estructura del notebook:**
1. Problema y datos (15%)
2. Modelado y análisis (35%)
3. Interpretación con SHAP (25%)
4. Conclusiones (15%)
5. Uso de IA (10%)

---
## 1. Problema y datos
### 1.1 Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ruta al dataset desde la carpeta notebooks/
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw', 'wingspan-20260128.xlsx')

# Las filas 1 y 2 del Excel contienen estadísticas internas del archivo, no cartas.
# skiprows=[1, 2] las ignora conservando el header real en la fila 0.
df_raw = pd.read_excel(DATA_PATH, sheet_name='Birds', skiprows=[1, 2])

print(f'Dimensiones del dataset: {df_raw.shape}')
df_raw.head(3)

### 1.2 Exploración inicial

In [ ]:
# Tipos de datos por columna
df_raw.dtypes

In [ ]:
# Valores nulos por columna
nulls = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(1)
null_summary = pd.DataFrame({'nulos': nulls, 'porcentaje': nulls_pct})
null_summary[null_summary['nulos'] > 0].sort_values('porcentaje', ascending=False)

In [ ]:
# Estadísticas descriptivas de variables numéricas clave
df_raw[['Victory points', 'Total food cost', 'Egg limit', 'Wingspan']].describe()

In [ ]:
# Distribución por color (tipo de activación) y expansión
print('Distribución por color de poder:')
print(df_raw['Color'].value_counts())
print('\nDistribución por expansión (Set):')
print(df_raw['Set'].value_counts())

### 1.3 Limpieza de datos

In [ ]:
df = df_raw.copy()

# Eliminar columnas no relevantes para el modelo
cols_drop = [
    'Scientific name', 'Flavor text', 'Fan Art flavor text',
    'Fan Art Pack?', 'Fan art beak direction', 'Swift Start',
    'Automa ban', '/ (food cost)', '* (food cost)'
]
df.drop(columns=cols_drop, inplace=True)

# Columnas binarias marcadas con 'X': convertir a 1/0
binary_cols = [
    'Predator', 'Flocking', 'Bonus card',
    'Forest', 'Grassland', 'Wetland',
    'North America', 'Central America', 'South America',
    'Europe', 'Asia', 'Africa', 'Oceania'
]
for col in binary_cols:
    df[col] = df[col].apply(lambda x: 1 if x == 'X' else 0)

# Columnas de alimento: nulo significa que no requiere ese tipo → rellenar con 0
food_cols = ['Invertebrate', 'Seed', 'Fish', 'Fruit', 'Rodent', 'Nectar', 'Wild (food)']
df[food_cols] = df[food_cols].fillna(0)

# Eliminar filas sin nombre de ave (registros incompletos)
df.dropna(subset=['Common name'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Filas después de limpieza: {len(df)}')
print(f'Columnas restantes: {df.shape[1]}')

### 1.4 Variable objetivo: tipo de motor (`engine_type`)

La etiqueta de clasificación no existe en el dataset crudo y se deriva de los atributos de cada carta. Se usa una función de scoring ponderado basada en las reglas reales del juego (ver `.claude/skills/wingspan-domain.md`):

| Tipo de motor | Hábitat principal | Señales clave |
|---|---|---|
| `food` | Bosque | `Forest=1`, poder con "food" o "cache", costo bajo, acepta comodín |
| `egg` | Pradera | `Grassland=1`, poder con "egg" o "lay", `Egg limit ≥ 4` |
| `card` | Humedal | `Wetland=1`, poder con "draw", `Flocking=1`, `Predator=1`, VP alto |
| `neutral` | Sin señal dominante | Score total = 0 |

**Limitación estadística:** esta derivación introduce sesgo de diseño (*label leakage* conceptual). La etiqueta ideal vendría de datos reales de partidas. Se documenta como limitación en las conclusiones.

In [ ]:
def classify_engine(row):
    """Clasifica una carta según el motor al que contribuye.
    Usa scoring ponderado basado en las reglas de Wingspan.
    Fuente: .claude/skills/wingspan-domain.md
    """
    scores = {'food': 0, 'egg': 0, 'card': 0}
    power = str(row.get('Power text', '')).lower()

    # Señales de motor de alimento (Bosque)
    if row.get('Forest', 0) == 1:
        scores['food'] += 2
    if any(kw in power for kw in ['food', 'cache', 'get food']):
        scores['food'] += 3
    if row.get('Wild (food)', 0) > 0:
        scores['food'] += 1
    if row.get('Total food cost', 99) <= 1:
        scores['food'] += 1

    # Señales de motor de huevos (Pradera)
    if row.get('Grassland', 0) == 1:
        scores['egg'] += 2
    if any(kw in power for kw in ['egg', 'lay']):
        scores['egg'] += 3
    if row.get('Egg limit', 0) >= 4:
        scores['egg'] += 2

    # Señales de motor de robo de cartas (Humedal)
    if row.get('Wetland', 0) == 1:
        scores['card'] += 2
    if any(kw in power for kw in ['draw', 'tuck', 'card']):
        scores['card'] += 3
    if row.get('Flocking', 0) == 1:
        scores['card'] += 2
    if row.get('Predator', 0) == 1:
        scores['card'] += 1
    if row.get('Victory points', 0) >= 4:
        scores['card'] += 1

    dominant = max(scores, key=scores.get)
    return dominant if scores[dominant] > 0 else 'neutral'

df['engine_type'] = df.apply(classify_engine, axis=1)

print('Distribución de tipos de motor:')
print(df['engine_type'].value_counts())
print(f'\nPorcentajes:')
print((df['engine_type'].value_counts() / len(df) * 100).round(1))

### 1.5 Visualización exploratoria

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploración del dataset Wingspan', fontsize=14, fontweight='bold')

# Gráfico 1: distribución de tipos de motor
engine_counts = df['engine_type'].value_counts()
colors_motor = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
axes[0].barh(engine_counts.index, engine_counts.values, color=colors_motor)
axes[0].set_title('Distribución de tipos de motor')
axes[0].set_xlabel('Cantidad de cartas')

# Gráfico 2: distribución de puntos de victoria
axes[1].hist(df['Victory points'].dropna(), bins=10, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de puntos de victoria')
axes[1].set_xlabel('Victory points')
axes[1].set_ylabel('Frecuencia')

# Gráfico 3: proporción de hábitats por tipo de motor
habitat_by_engine = df.groupby('engine_type')[['Forest', 'Grassland', 'Wetland']].mean()
habitat_by_engine.plot(kind='bar', ax=axes[2], colormap='Set2')
axes[2].set_title('Proporción de hábitats por tipo de motor')
axes[2].set_xlabel('')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30, ha='right')
axes[2].set_ylabel('Proporción promedio')

plt.tight_layout()
plt.show()

**Interpretación:** La distribución de tipos de motor refleja el diseño del juego: las cartas de motor de alimento y huevos son las más frecuentes porque el juego base está centrado en Bosque y Pradera. El motor de cartas (Humedal) es menos común en el juego base pero crece con las expansiones. Las cartas neutras son aquellas con poderes pasivos o de un solo uso que no contribuyen a ningún motor de forma clara.

---
## 2. Modelado y análisis

*Esta sección se completa en el siguiente commit.*

---
## 3. Interpretación con SHAP

*Esta sección se completa en el siguiente commit.*

---
## 4. Conclusiones

*Esta sección se completa en el siguiente commit.*

---
## 5. Uso de IA

*Esta sección se completa en el siguiente commit.*